# Processing the initial gold standard set

We apply some filter to account only for annotated emission records that are present in table or text formats. Here we omit any emission records present in the original reports that are displayed in any illustration formats.

In [21]:
from pathlib import Path
import pandas as pd
import yaml
import json

## Load original datasets

In [22]:
PDF_DIR = "../data/reports"
DATA_DIR = Path("../data/gold_data")

## Initial Preprocessing - Filtering and Data Cleaning

**ATTENTION**

*It was noticed that the original dataset contained incorrect page numbers that did not match the original page numbering in the real PDFs. We validated the page numbers carefully and updated them in the original dataset accordingly.*

- Remove all entries with missing page numbers 
    - **Reason**: Account only for reports that contain emission records
- Remove entry with invalid page number "Env33"
- Filter *display type* to only consider `tables` and `text`
    - **Reason**: We do not consider emission records that are presented in graphical illustrations (#TODO with VLM)

In [23]:
df = pd.read_csv(DATA_DIR / "gold_standard_origin.csv")
df.insert(0, "report_id", df["company_name"].astype(str) + "_" + df["report_year"].astype(str))
len(df["report_id"].unique())

139

In [24]:
# Filter the dataframe to only include rows where the report_id is in both the original dataframe and the list of company reports
companies_in_df = df["report_id"].unique().tolist()
company_reports = [r.stem for r in Path(PDF_DIR).rglob("*.pdf")]

companies_not_in_both = list(set(companies_in_df) - set(company_reports))
df = df[df["report_id"].isin(companies_not_in_both) == False]
print(len(df["report_id"].unique()))
df

139


,report_id,report_name,company_name,report_year,report_type,url,year,scope,page,value,unit,unit_normalized,metric_name,display_type,extracted_text_from_page,merge_id
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,https://www.allianz.com/content/dam/onemarketi...,2013,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,https://www.allianz.com/content/dam/onemarketi...,2014,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,https://www.allianz.com/content/dam/onemarketi...,2015,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,https://www.allianz.com/content/dam/onemarketi...,2016,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,https://www.allianz.com/content/dam/onemarketi...,2017,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5626,xvivo perfusion_2021,xvivo perfusion_2021_report.pdf,xvivo perfusion,2021,Annual Report,https://storage.mfn.se/6f621ebc-3799-4048-8439...,2018,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5640
5627,xvivo perfusion_2021,xvivo perfusion_2021_report.pdf,xvivo perfusion,2021,Annual Report,https://storage.mfn.se/6f621ebc-3799-4048-8439...,2019,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5641
5628,xvivo perfusion_2021,xvivo perfusion_2021_report.pdf,xvivo perfusion,2021,Annual Report,https://storage.mfn.se/6f621ebc-3799-4048-8439...,2020,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5642
5629,xvivo perfusion_2021,xvivo perfusion_2021_report.pdf,xvivo perfusion,2021,Annual Report,https://storage.mfn.se/6f621ebc-3799-4048-8439...,2021,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5643


In [25]:
# All report IDs in the original dataframe
all_reports = set(df["report_id"].dropna().unique())

# Reports without GHG data
reports_without_data = set(
    df.groupby("report_id")["page"]
    .apply(lambda pages: pages.isna().all())
    .loc[lambda x: x]
    .index
)

# Reports with GHG data
reports_with_data = all_reports - reports_without_data

print("Original reports:", len(all_reports))
print("Reports without data:", len(reports_without_data))
print("Reports with data:", len(reports_with_data))

Original reports: 139
Reports without data: 69
Reports with data: 70


In [26]:
df.dropna(subset=["page"], inplace=True)
df = df[df["page"].isin(["Env33"]) == False]

df["page"] = df["page"].astype(str).str.replace("*", "") # Remove asterisks from page numbers
df = df.astype({"page": int}) # Convert page numbers to integers
display_type = df['display_type'].unique().tolist()

df = df[df["display_type"].isin(["Table", "Text/Table", "Table/Text", "Text"])] # filter only tables and text

df.drop(columns=["extracted_text_from_page", "merge_id", "url"], inplace=True)
print(len(df["report_id"].unique()))
df

64


C:\Users\Marvin.Klein\AppData\Local\Temp\ipykernel_28768\1859819062.py:1: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=["page"], inplace=True)
C:\Users\Marvin.Klein\AppData\Local\Temp\ipykernel_28768\1859819062.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["extracted_text_from_page", "merge_id", "url"], inplace=True)


,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type
7,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,2020,1,79,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
9,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,2021,1,79,28699.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
11,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,2022,1,79,30953.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
20,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,2020,2lb,79,180826.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
21,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022,Sustainability Report,2021,2lb,79,149599.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5486,webuild_2019,webuild_2019_report.pdf,webuild,2019,other,2018,2lb,44,54637.0,tCO2e,t CO2e,Indirect emissions (Scope 2),Table
5487,webuild_2019,webuild_2019_report.pdf,webuild,2019,other,2019,2lb,44,43371.0,tCO2e,t CO2e,Indirect emissions (Scope 2),Table
5505,webuild_2019,webuild_2019_report.pdf,webuild,2019,other,2017,3,44,137577.0,tCO2e,t CO2e,Other indirect emissions (Scope 3)/Other indir...,Table
5506,webuild_2019,webuild_2019_report.pdf,webuild,2019,other,2018,3,44,117528.0,tCO2e,t CO2e,Other indirect emissions (Scope 3)/Other indir...,Table


In [28]:
new_row = pd.DataFrame({"report_id": list(reports_without_data), "company_name": [entry.split("_")[0] for entry in reports_without_data]})

df = pd.concat([df, new_row], ignore_index=True)
print("Final dataset:", len(df["report_id"].unique()))
df

Final dataset: 133


,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,1,79.0,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
1,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,1,79.0,28699.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
2,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022.0,1,79.0,30953.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
3,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,2lb,79.0,180826.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
4,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,2lb,79.0,149599.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
...,...,...,...,...,...,...,...,...,...,...,...,...,...
680,suntec_2017,NaN,suntec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
681,beach energy ltd_2020,NaN,beach energy ltd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
682,trex inc_2019,NaN,trex inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
683,first bancshares inc_2017,NaN,first bancshares inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Scope renaming to match our canonical scope names
scope_mapping = {"1": "Scope 1",
                 "2lb": "Scope 2 (location based)",
                 "2mb": "Scope 2 (market based)",
                 "3": "Scope 3"}

df["scope"] = df["scope"].map(scope_mapping)
df

,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,Scope 1,79.0,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
1,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,Scope 1,79.0,28699.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
2,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022.0,Scope 1,79.0,30953.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
3,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,Scope 2 (location based),79.0,180826.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
4,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,Scope 2 (location based),79.0,149599.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
...,...,...,...,...,...,...,...,...,...,...,...,...,...
680,suntec_2017,NaN,suntec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
681,beach energy ltd_2020,NaN,beach energy ltd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
682,trex inc_2019,NaN,trex inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
683,first bancshares inc_2017,NaN,first bancshares inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Remove emission record duplicates

Because the original gold standard dataset contains multiple entries for the emission record of the same company in the same year, we omit these duplicates.

**Reason**: Even though insightful for subsequent analyses, dropping duplicates supports the extraction task and simplifies the processing of context using LLMs in the generation module.

In [30]:
df.drop_duplicates(subset=["report_id", "company_name", "year", "scope"], inplace=True)
df

C:\Users\Marvin.Klein\AppData\Local\Temp\ipykernel_28768\584286460.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(subset=["report_id", "company_name", "year", "scope"], inplace=True)


,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,Scope 1,79.0,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
1,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,Scope 1,79.0,28699.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
2,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022.0,Scope 1,79.0,30953.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
3,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,Scope 2 (location based),79.0,180826.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
4,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,Scope 2 (location based),79.0,149599.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,suntec_2017,NaN,suntec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
612,beach energy ltd_2020,NaN,beach energy ltd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
613,trex inc_2019,NaN,trex inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
614,first bancshares inc_2017,NaN,first bancshares inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Perfrom unit normalization

We observe some inconsistencies in the `normalized_unit` column of the original dataset.

Therefore, we perform normalization.

In [31]:
df['unit_normalized'].unique().tolist()

['t CO2e',
 'kt',
 'CO2 emissions from energy consumption (in 1,000 t)',
 'kt CO2e',
 'Mt CO2e',
 'MT CO2e',
 'kg CO2e',
 'ton CO2e',
 'MM MT CO2e',
 'CO2 emissions (1,000 tonnes)',
 'metric tonnes of CO2 equivalent',
 'CO2e (US tons)',
 't',
 'm t CO2e',
 'CO2e in t',
 'emissions',
 'kt CO2',
 'ktons CO2',
 'CO2e emissions (tonnes)',
 'mt CO2e',
 'CO2e t',
 'million t',
 'million t CO2',
 nan]

In [33]:
unit_norm_df = pd.read_csv(DATA_DIR / "unit_normalization_dict.csv")
unit_mapping = dict(zip(unit_norm_df['unit'], unit_norm_df['normalized_unit']))
df['unit_normalized'] = df['unit'].map(unit_mapping)
df['unit_normalized'].unique().tolist()
df = df.astype({"page": "Int64"})
df

,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,Scope 1,79,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
1,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,Scope 1,79,28699.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
2,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022.0,Scope 1,79,30953.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table
3,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020.0,Scope 2 (location based),79,180826.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
4,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021.0,Scope 2 (location based),79,149599.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table
...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,suntec_2017,NaN,suntec,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN
612,beach energy ltd_2020,NaN,beach energy ltd,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN
613,trex inc_2019,NaN,trex inc,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN
614,first bancshares inc_2017,NaN,first bancshares inc,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN


# Combine with evidence text

In the next step, the processed emission records are combined with the evidence text present in the report. 

The evidence (or extracted) text are received by utilizing `Docling` as preprocessing PDF conversion, that preserves report content and layout.

The evidence (or extracted) text are presented in `Markdown` format.

**Note**: The entire page content is used as evidence text. This ensures that not only the (tabular) information of emission records are present, but also surrounding layout elements such as `footnotes`, `headers`, and other textual information.

In [38]:
from docling_core.types.doc.document import DoclingDocument, ContentLayer
from docling_core.transforms.serializer.markdown import MarkdownDocSerializer, MarkdownParams

In [39]:
OUTPUT_DIR = "../output/converted"

def load_json_file(report_id) -> DoclingDocument:
    """
    Loads the JSON file for a given company as the parsed `DoclingDocument`.
    
    :param report_id: The name of the report (company name_yyyy) for which to load the `JSON` file.
    :return: A `DoclingDocument` object representing the parsed document.
    """
    company = report_id.split("_")[0]
    json_files = list((Path(OUTPUT_DIR) / company).glob("*.json"))
    if len(json_files) == 0:
        raise FileNotFoundError(f"No JSON file found for company '{company}'")
    json_path = [json_files[i] for i, f in enumerate(json_files) if f.stem == report_id][0]
    doc_dl = DoclingDocument.load_from_json(json_path)

    return doc_dl

In [41]:
df["page"] = df["page"].astype("Int64")
df["year"] = df["year"].astype("Int64")

for report_id in df["report_id"].unique():
    for page_no in df[df["report_id"] == report_id]["page"].dropna().unique():
        #if page_no.isna():
            
        doc_dl = load_json_file(report_id)
        serializer = MarkdownDocSerializer(doc=doc_dl, params=MarkdownParams(
            layers={ContentLayer.BODY, ContentLayer.FURNITURE},
            pages={page_no},
            include_hyperlinks=False,
            traverse_pictures=False,
            image_placeholder=" ",),)
        
        md = serializer.serialize().text
        df.loc[(df["report_id"] == report_id) & (df["page"] == page_no), "evidence_text"] = md
    
df

,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type,evidence_text
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020,Scope 1,79,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table,01 Introduction and strategy\n\n02 Measuring a...
1,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021,Scope 1,79,28699.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table,01 Introduction and strategy\n\n02 Measuring a...
2,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022,Scope 1,79,30953.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table,01 Introduction and strategy\n\n02 Measuring a...
3,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020,Scope 2 (location based),79,180826.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table,01 Introduction and strategy\n\n02 Measuring a...
4,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021,Scope 2 (location based),79,149599.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (location based),Table,01 Introduction and strategy\n\n02 Measuring a...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,suntec_2017,NaN,suntec,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
612,beach energy ltd_2020,NaN,beach energy ltd,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
613,trex inc_2019,NaN,trex inc,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
614,first bancshares inc_2017,NaN,first bancshares inc,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
df.sort_values("report_id", inplace=True)
df

C:\Users\Marvin.Klein\AppData\Local\Temp\ipykernel_28768\1489721240.py:1: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_values("report_id", inplace=True)


,report_id,report_name,company_name,report_year,report_type,year,scope,page,value,unit,unit_normalized,metric_name,display_type,evidence_text
0,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2020,Scope 1,79,28714.0,t CO2e,t CO2e,Scope 1 – Direct GHG emissions,Table,01 Introduction and strategy\n\n02 Measuring a...
11,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022,Scope 3,79,92467.0,t CO2e,t CO2e,Scope 3 – Other indirect GHG emissions,Table,01 Introduction and strategy\n\n02 Measuring a...
10,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021,Scope 3,79,55359.0,t CO2e,t CO2e,Scope 3 – Other indirect GHG emissions,Table,01 Introduction and strategy\n\n02 Measuring a...
8,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2022,Scope 2 (market based),79,30490.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (market based...,Table,01 Introduction and strategy\n\n02 Measuring a...
7,Allianz_2022,Allianz_2022_report.pdf,Allianz,2022.0,Sustainability Report,2021,Scope 2 (market based),79,54689.0,t CO2e,t CO2e,Scope 2 – Indirect GHG emissions (market based...,Table,01 Introduction and strategy\n\n02 Measuring a...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,webuild_2019,webuild_2019_report.pdf,webuild,2019.0,other,2018,Scope 3,44,117528.0,tCO2e,t CO2e,Other indirect emissions (Scope 3)/Other indir...,Table,"| Energy consumption (GRI 302-1, 302-2, 302-3)..."
540,webuild_2019,webuild_2019_report.pdf,webuild,2019.0,other,2019,Scope 1,44,350593.0,tCO2e,t CO2e,Direct emissions (Scope 1),Table,"| Energy consumption (GRI 302-1, 302-2, 302-3)..."
593,western union_2019,NaN,western union,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
564,xvivo perfusion_2020,NaN,xvivo perfusion,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN


## Save the processed dataset as CSV file

In [43]:
df.to_csv(DATA_DIR / "gold_standard_processed.csv", index=False)